<a href="https://colab.research.google.com/github/Kienknu/Kienknu/blob/main/Chemical_space_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install rdkit
!pip install molmass

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.1 MB/s eta 0:00:00


In [2]:
import rdkit
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Draw, rdChemReactions
from rdkit.Chem.Draw import IPythonConsole
from IPython.display import display
import pandas as pd
import re

**1. Uploading SMARTS rules**

In [10]:
import pandas as pd

# 1. Prompt the user for the Excel file path
reaction_rule_file = "Reaction_rules_new_Geondo.xlsx" # sellect data file

# 2. Read all sheet names from the reaction_rule_file
try:
    xls = pd.ExcelFile(reaction_rule_file)
    sheet_names = xls.sheet_names

    if not sheet_names:
        print(f"Error: No sheets found in '{reaction_rule_file}'.")
    else:
        # 3. Present the list of sheet names to the user
        print("\nAvailable sheets:")
        for i, sheet_name in enumerate(sheet_names):
            print(f"{i+1}: {sheet_name}")

        selected_sheet_name = None
        while selected_sheet_name is None:
            try:
                # 4. Prompt the user to enter the number corresponding to the sheet
                selection = int(input("Enter the number of the sheet you want to select: "))

                # 5. Validate the user's input
                if 1 <= selection <= len(sheet_names):
                    selected_sheet_name = sheet_names[selection - 1]
                    print(f"\nSelected sheet: '{selected_sheet_name}'")
                else:
                    print(f"Invalid selection. Please enter a number between 1 and {len(sheet_names)}.")
            except ValueError:
                print("Invalid input. Please enter a number.")

        # 6. Load the data from the selected sheet into a pandas DataFrame
        if selected_sheet_name:
            reaction_rules_df = pd.read_excel(reaction_rule_file, sheet_name=selected_sheet_name)
            print(f"Successfully loaded sheet '{selected_sheet_name}' into 'reaction_rule_file'.")
            print("First 5 rows of the loaded DataFrame:")
            display(reaction_rules_df.head())

except FileNotFoundError:
    print(f"Error: The file '{reaction_rule_file}' was not found. Please ensure the full file path including extension is correct.")
except Exception as e:
    print(f"An error occurred: {e}")


#-------Converting reaction into SMARTS reaction rules----#
reactant_product_pairs = []

# Check if the DataFrame exists and has the expected columns
if reaction_rules_df is not None and 'Reactant' in reaction_rules_df.columns and 'Product' in reaction_rules_df.columns:
    for index, row in reaction_rules_df.iterrows():
        reactant_smiles = row['Reactant']
        product_smiles = row['Product']
        # Append the reactant and product SMILES as a tuple to the list
        reactant_product_pairs.append((reactant_smiles, product_smiles))

    print(f"Extracted {len(reactant_product_pairs)} reactant-product pairs.")
    # Display the first few extracted pairs to verify
    if reactant_product_pairs:
        print("First 5 extracted pairs:")
        for i, pair in enumerate(reactant_product_pairs[:5]):
            print(f"  Pair {i+1}: Reactant='{pair[0]}', Product='{pair[1]}'")
else:
    print("DataFrame 'reaction_rules_df' is not available or does not contain 'Reactant' and 'Product' columns.")

#-------------RDKit reaction rules-----#
reaction_rules = []
# Iterate through the extracted reactant-product pairs
for reactant_smiles, product_smiles in reactant_product_pairs:
    # Ensure neither reactant nor product is None or NaN
    if pd.notna(reactant_smiles) and pd.notna(product_smiles):
        try:
            # Create the reaction SMARTS string
            reaction_smarts = f"{reactant_smiles}>>{product_smiles}"
            # Create the RDKit reaction object
            rxn = AllChem.ReactionFromSmarts(reaction_smarts)
            # Append the reaction object to the list
            reaction_rules.append(rxn)
        except Exception as e:
            # Print a warning if a rule could not be created
            print(f"Warning: Could not create reaction rule from '{reactant_smiles}>>{product_smiles}': {e}")
    else:
        print(f"Warning: Skipping row due to missing reactant or product SMILES: Reactant='{reactant_smiles}', Product='{product_smiles}'")

print(f"Successfully created {len(reaction_rules)} RDKit reaction rules.")


Available sheets:
1: Selected_rearranged_Paper_2
Enter the number of the sheet you want to select: 1

Selected sheet: 'Selected_rearranged_Paper_2'
Successfully loaded sheet 'Selected_rearranged_Paper_2' into 'reaction_rule_file'.
First 5 rows of the loaded DataFrame:


,No,Reaction,Reactant,Product
0,0,Hydroxylation,[#6h:1],[*:1]O
1,1,Methylene Oxidation,[#6h2:1],[*:1]=O
2,2,Dealkylation,"[#6H2:1][#7,#8H0,#16:2]",[*:2].[*:1](=O)O
3,3,NaN,"[#6H2:1][#7,#8H0,#16:2]",[*:2].[*:1]=O
4,4,NaN,"[#6H2:1][#7,#8H0,#16:2]",[*:2].[*:1]-O


Extracted 40 reactant-product pairs.
First 5 extracted pairs:
  Pair 1: Reactant='[#6h:1]', Product='[*:1]O'
  Pair 2: Reactant='[#6h2:1]', Product='[*:1]=O'
  Pair 3: Reactant='[#6H2:1][#7,#8H0,#16:2]', Product='[*:2].[*:1](=O)O'
  Pair 4: Reactant='[#6H2:1][#7,#8H0,#16:2]', Product='[*:2].[*:1]=O'
  Pair 5: Reactant='[#6H2:1][#7,#8H0,#16:2]', Product='[*:2].[*:1]-O'
Successfully created 40 RDKit reaction rules.


**2. Loading a reaction transformation file (e.g., MCM)**

In [18]:
import pandas as pd

# 1. Loading the Excel file
Initial_chemical_space_file = 'Initial_chemical_space_Geondo.xlsx'

# 2. Read all sheet names from the Initial_chemical_space_file
try:
    xls = pd.ExcelFile(Initial_chemical_space_file)
    sheet_names = xls.sheet_names

    if not sheet_names:
        print(f"Error: No sheets found in '{Initial_chemical_space_file}'.")
    else:
        # 3. Present the list of sheet names to the user
        print("\nAvailable sheets:")
        for i, sheet_name in enumerate(sheet_names):
            print(f"{i+1}: {sheet_name}")

        selected_sheet_name = None
        while selected_sheet_name is None:
            try:
                # 4. Prompt the user to enter the number corresponding to the sheet
                selection = int(input("Enter the number of the sheet you want to select: "))

                # 5. Validate the user's input
                if 1 <= selection <= len(sheet_names):
                    selected_sheet_name = sheet_names[selection - 1]
                    print(f"\nSelected sheet: '{selected_sheet_name}'")
                else:
                    print(f"Invalid selection. Please enter a number between 1 and {len(sheet_names)}.")
            except ValueError:
                print("Invalid input. Please enter a number.")

        # 6. Load the data from the selected sheet into a pandas DataFrame
        if selected_sheet_name:
            Initial_chemical_space_file_df = pd.read_excel(Initial_chemical_space_file, sheet_name=selected_sheet_name)
            print(f"Successfully loaded sheet '{selected_sheet_name}' into 'Initial_chemical_space_file'.")
            print("First 5 rows of the loaded DataFrame:")
            display(Initial_chemical_space_file_df.head())


except FileNotFoundError:
    print(f"Error: The file '{Initial_chemical_space_file}' was not found. Please ensure the full file path including extension is correct.")
except Exception as e:
    print(f"An error occurred: {e}")


# II. Initialize an empty list to store the reaction steps
Initial_reaction_history = []

if not Initial_chemical_space_file_df.empty and 'Reactant' in Initial_chemical_space_file_df.columns and 'Product' in Initial_chemical_space_file_df.columns:
    for index, row in Initial_chemical_space_file_df.iterrows():
        reactant_smiles = row['Reactant']
        product_smiles = row['Product']

        # Ensure neither reactant nor product SMILES are NaN and are strings
        if pd.notna(reactant_smiles) and pd.notna(product_smiles):
            reactant_smi_str = str(reactant_smiles)
            product_smi_str = str(product_smiles)

            # Canonicalize reactant SMILES once
            canonical_reactant_smi = None
            try:
                mol_reactant = Chem.MolFromSmiles(reactant_smi_str)
                if mol_reactant is not None:
                    canonical_reactant_smi = Chem.MolToSmiles(mol_reactant, canonical=True)
            except Exception as e:
                print(f"Warning: Could not canonicalize reactant SMILES '{reactant_smi_str}' in row {index}: {e}")
                continue

            if canonical_reactant_smi is None:
                continue

            # Handle multi-component products
            if '.' in product_smi_str:
                individual_product_smiles = product_smi_str.split('.')
                for prod_smi in individual_product_smiles:
                    try:
                        mol_product = Chem.MolFromSmiles(prod_smi)
                        if mol_product is not None:
                            canonical_product_smi = Chem.MolToSmiles(mol_product, canonical=True)
                            Initial_reaction_history.append((canonical_reactant_smi, index, canonical_product_smi))
                        else:
                            print(f"Warning: Invalid individual product SMILES '{prod_smi}' in row {index}. Skipping.")
                    except Exception as e:
                        print(f"Warning: Could not canonicalize individual product SMILES '{prod_smi}' in row {index}: {e}. Skipping.")
            else:
                # Existing logic for single product
                try:
                    mol_product = Chem.MolFromSmiles(product_smi_str)
                    if mol_product is not None:
                        canonical_product_smi = Chem.MolToSmiles(mol_product, canonical=True)
                        Initial_reaction_history.append((canonical_reactant_smi, index, canonical_product_smi))
                    else:
                        print(f"Warning: Invalid product SMILES '{product_smi_str}' in row {index}. Skipping.")
                except Exception as e:
                    print(f"An error occurred while canonicalizing product SMILES '{product_smi_str}' in row {index}: {e}. Skipping.")
    print(f"Extracted {len(Initial_reaction_history)} canonicalized reaction steps into Initial_reaction_history.")
else:
    print("No reaction rules to process or missing 'Reactant'/'Product' columns in the Excel sheet.")

# Display the first entries in the excel_reaction_history
print("First 5 entries in Initial_reaction_history:")
for entry in Initial_reaction_history[:5]:
    print(entry)

print(60*"-")
#----Generating an initial chemical space----#
all_unique_smiles_from_rules = set()

for entry in Initial_reaction_history:
    reactant_smi = entry[0]
    product_smi = entry[2]

    all_unique_smiles_from_rules.add(reactant_smi)
    all_unique_smiles_from_rules.add(product_smi)

# Set the initial_chemical_space to include all these unique molecules
initial_chemical_space = all_unique_smiles_from_rules

print(f"New chemical space initialized with {len(initial_chemical_space)} unique molecules from all reactants and products in the Excel rules.")


Available sheets:
1: Oleic acid
2: Palmitoleic acid
3: linoleic acid
4: conjugated linoleic acid
5: Elaidic acid
Enter the number of the sheet you want to select: 1

Selected sheet: 'Oleic acid'
Successfully loaded sheet 'Oleic acid' into 'Initial_chemical_space_file'.
First 5 rows of the loaded DataFrame:


,Reactant,Product
0,CCCCCCCC\C=C/CCCCCCCC(O)=O,CCCCCCCC\C=C/CCCCCCCC(O)=O


Extracted 1 canonicalized reaction steps into Initial_reaction_history.
First 5 entries in Initial_reaction_history:
('CCCCCCCC/C=C\\CCCCCCCC(=O)O', 0, 'CCCCCCCC/C=C\\CCCCCCCC(=O)O')
------------------------------------------------------------
New chemical space initialized with 1 unique molecules from all reactants and products in the Excel rules.


**3. Split Initial Chemical Space into Chunks**

In [19]:
chunk_size = int(input("Enter the maximum number of compounds for each chemical space chunk: "))
print(f"Chemical space chunk size set to: {chunk_size}")
print(60*'-')

chemical_space_list = sorted(list(initial_chemical_space))
chemical_space_chunks = []

for i in range(0, len(chemical_space_list), chunk_size):
    chemical_space_chunks.append(chemical_space_list[i:i + chunk_size])

print(f"Total number of chunks created: {len(chemical_space_chunks)}")
if chemical_space_chunks:
    print(f"Size of the first chunk: {len(chemical_space_chunks[0])}")
    print(f"Size of the last chunk: {len(chemical_space_chunks[-1])}")
else:
    print("No chunks were created as chemical_space was empty.")

Enter the maximum number of compounds for each chemical space chunk: 1
Chemical space chunk size set to: 1
------------------------------------------------------------
Total number of chunks created: 1
Size of the first chunk: 1
Size of the last chunk: 1


**4. Generating chemical space**

In [21]:
print(f"There are {len(chemical_space_chunks)} chunks available (indices 0 to {len(chemical_space_chunks) - 1}).")
selected_chunk_index = int(input("Enter the index of the chemical space chunk you want to simulate: "))

# Validate the input index
if 0 <= selected_chunk_index < len(chemical_space_chunks):
    initial_chemical_space_chunk = chemical_space_chunks[selected_chunk_index]
    print(f"\nSelected chunk {selected_chunk_index} (size: {len(initial_chemical_space_chunk)} compounds).")

    # Initialize variables for this specific chunk's simulation
    selected_chunk_sim_space = set(initial_chemical_space_chunk)
    selected_chunk_sim_history = []

    num_iterations = 2

    for iteration in range(num_iterations):
        temp_new_products_for_chunk = set()
        current_chunk_list = list(selected_chunk_sim_space)

        for reactant_smiles in current_chunk_list:
            reactant_mol = Chem.MolFromSmiles(reactant_smiles)
            if reactant_mol:
                for rxn_index, rxn in enumerate(reaction_rules):
                    try:
                        num_reactants_in_rule = rxn.GetNumReactantTemplates()

                        if num_reactants_in_rule == 1:
                            possible_products = rxn.RunReactants((reactant_mol,))
                            for prod_set in possible_products:
                                for prod in prod_set:
                                    try:
                                        prod_smiles = Chem.MolToSmiles(prod)
                                        individual_prod_smiles = prod_smiles.split('.')

                                        for single_prod_smi in individual_prod_smiles:
                                            prod_mol_normalized = Chem.MolFromSmiles(single_prod_smi)
                                            if prod_mol_normalized is not None:
                                                prod_smiles_canonical = Chem.MolToSmiles(prod_mol_normalized, canonical=True)

                                                prod_mol = Chem.MolFromSmiles(prod_smiles_canonical,sanitize=False)
                                                if prod_mol is not None and prod_mol.GetNumAtoms() > 0:
                                                  try:
                                                    Chem.rdmolops.SanitizeMol(prod_mol, Chem.rdmolops.SanitizeFlags.SANITIZE_ALL)
                                                  except:
                                                    pass

                                                  if len(Chem.DetectChemistryProblems(prod_mol)) == 0:
                                                    if prod_smiles_canonical not in selected_chunk_sim_space:
                                                        temp_new_products_for_chunk.add(prod_smiles_canonical)
                                                        selected_chunk_sim_history.append((reactant_smiles, rxn_index, prod_smiles_canonical))
                                    except Exception as e:
                                        pass

                        elif num_reactants_in_rule > 1:
                            reactant_templates = [rxn.GetReactantTemplate(i) for i in range(num_reactants_in_rule)]
                            all_mols_in_chunk = [(Chem.MolFromSmiles(smi), smi) for smi in current_chunk_list if Chem.MolFromSmiles(smi) is not None]

                            import itertools
                            for combo in itertools.combinations(all_mols_in_chunk, num_reactants_in_rule):
                                mols_in_combo = [item[0] for item in combo]
                                smiles_in_combo = [item[1] for item in combo]

                                is_match = True
                                for j in range(num_reactants_in_rule):
                                    if not mols_in_combo[j].HasSubstructMatch(reactant_templates[j]):
                                        is_match = False
                                        break

                                if is_match:
                                    possible_products = rxn.RunReactants(tuple(mols_in_combo))
                                    for prod_set in possible_products:
                                        for prod in prod_set:
                                            try:
                                                prod_smiles = Chem.MolToSmiles(prod)
                                                individual_prod_smiles = prod_smiles.split('.')

                                                for single_prod_smi in individual_prod_smiles:
                                                    prod_mol_normalized = Chem.MolFromSmiles(single_prod_smi)
                                                    if prod_mol_normalized is not None:
                                                        prod_smiles_canonical = Chem.MolToSmiles(prod_mol_normalized, canonical=True)

                                                        prod_mol = Chem.MolFromSmiles(prod_smiles_canonical,sanitize=False)
                                                        if prod_mol is not None and prod_mol.GetNumAtoms() > 0:
                                                          try:
                                                            Chem.rdmolops.SanitizeMol(prod_mol, Chem.rdmolops.SanitizeFlags.SANITIZE_ALL)
                                                          except:
                                                            pass

                                                          if len(Chem.DetectChemistryProblems(prod_mol)) == 0:
                                                              if prod_smiles_canonical not in selected_chunk_sim_space:
                                                                  temp_new_products_for_chunk.add(prod_smiles_canonical)
                                                                  selected_chunk_sim_history.append((tuple(smiles_in_combo), rxn_index, prod_smiles_canonical))
                                            except Exception as e:
                                                pass
                    except Exception as e:
                        pass

        selected_chunk_sim_space.update(temp_new_products_for_chunk)

    selected_chunk_sim_space = {smi for smi in selected_chunk_sim_space if Chem.MolFromSmiles(smi) is not None}

    print(f"\nSimulation results for selected chunk {selected_chunk_index}:")
    print(f"  Chemical space size: {len(selected_chunk_sim_space)}")
    print(f"  Reaction history size: {len(selected_chunk_sim_history)}")

    #----- Save results of selected chunk-----#
    import json
    import os

    # Define filenames based on the selected chunk index
    sim_space_filename = f'simulated_chemical_space_chunk_{selected_chunk_index}_{sheet_name}.txt' #change its name
    sim_history_filename = f'simulated_reaction_history_chunk_{selected_chunk_index}_{sheet_name}.json'#change its name

    # Save the chemical space (set of SMILES) to a text file
    with open(sim_space_filename, 'w') as f:
        for smi in selected_chunk_sim_space:
            f.write(smi + '\n')
    print(f"Successfully saved simulated chemical space for chunk {selected_chunk_index} to '{sim_space_filename}'.")

    serializable_history = []
    for reactant, rxn_index, product in selected_chunk_sim_history:
        if isinstance(reactant, tuple):
            reactant = [str(r) for r in reactant]
        serializable_history.append((reactant, rxn_index, product))

    with open(sim_history_filename, 'w') as f:
        json.dump(serializable_history, f, indent=4)
    print(f"Successfully saved simulated reaction history for chunk {selected_chunk_index} to '{sim_history_filename}'.")

else:
    print(f"Error: Invalid chunk index. Please enter a number between 0 and {len(chemical_space_chunks) - 1}.")

There are 1 chunks available (indices 0 to 0).
Enter the index of the chemical space chunk you want to simulate: 0

Selected chunk 0 (size: 1 compounds).

Simulation results for selected chunk 0:
  Chemical space size: 4094
  Reaction history size: 8034
Successfully saved simulated chemical space for chunk 0 to 'simulated_chemical_space_chunk_0_Elaidic acid.txt'.
Successfully saved simulated reaction history for chunk 0 to 'simulated_reaction_history_chunk_0_Elaidic acid.json'.
